In [4]:
!pip install -q -U spacy
!python -m spacy download it_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 123.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import re
import spacy

nlp = spacy.load("it_core_news_sm")

# Supporto al modello small: aggiunge pattern PER per nome + cognome italiani
NOMI_COMUNI = [
    "alessandro", "andrea", "anna", "antonio", "chiara", "davide", "elena",
    "federica", "francesca", "francesco", "giorgio", "giovanni", "giulia",
    "giuseppe", "laura", "lorenzo", "luca", "luigi", "marco", "maria",
    "mario", "matteo", "paolo", "roberto", "sara", "silvia", "simone", "stefano"
]

if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
    ruler.add_patterns([
        {
            "label": "PER",
            "pattern": [
                {"LOWER": {"IN": NOMI_COMUNI}},
                {"LOWER": {"IN": ["de", "di", "del", "della", "lo", "la"]}, "OP": "?"},
                {"IS_TITLE": True, "IS_ALPHA": True},
            ],
        }
    ])

# Parole di apertura/chiusura che it_core_news_sm classifica erroneamente come PER
STOPWORDS_PER = {
    "saluti", "buongiorno", "buonasera", "cordiali", "gentile",
    "grazie", "ciao", "spett", "contatto", "richiesta"
}

REGEX_RULES = [
    (
        "CF",
        re.compile(
            r"(?<![A-Za-z0-9])[A-Za-z]{6}\d{2}[A-Za-z]\d{2}[A-Za-z]\d{3}[A-Za-z](?![A-Za-z0-9])"
        ),
    ),
    (
        "EMAIL",
        re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    ),
    (
        "TELEFONO",
        re.compile(
            r"(?<!\d)(?:(?:\+|00)39[\s.-]?)?(?:3\d{2}[\s.-]?\d{3}[\s.-]?\d{3,4}|0\d{1,3}[\s.-]?\d{3,4}[\s.-]?\d{3,4}|0\d{1,3}[\s.-]?\d{5,8})(?!\d)"
        ),
    ),
]


def anonymize_message(text: str) -> str:
    """Sostituisce CF, email, telefoni (regex) e nomi di persona (spaCy PER)."""
    if not text:
        return ""

    spans = []

    # 1. Estrazione tramite Regex
    for label, pattern in REGEX_RULES:
        for match in pattern.finditer(text):
            spans.append((match.start(), match.end(), f"[{label}]"))

    # 2. Estrazione entità PER tramite spaCy (escludendo stopwords e sovrapposizioni)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ == "PER" and ent.text.strip().lower() not in STOPWORDS_PER:
            overlaps = any(
                not (ent.end_char <= start or ent.start_char >= end)
                for start, end, _ in spans
            )
            if not overlaps:
                spans.append((ent.start_char, ent.end_char, "[NOME]"))

    # 3. Sostituzione da destra a sinistra per preservare gli indici
    spans.sort(key=lambda x: x[0], reverse=True)
    anonymized = text
    for start, end, placeholder in spans:
        anonymized = anonymized[:start] + placeholder + anonymized[end:]

    return anonymized


def build_gemini_payload(ticket: dict) -> dict:
    """
    Prepara il payload per Gemini:
    - Invia SOLO il messaggio anonimizzato.
    - Scarta tassativamente le note degli operatori.
    """
    raw_message = ticket.get("messaggio", "")
    return {
        "messaggio_anonimizzato": anonymize_message(raw_message)
    }

In [6]:
test_records = [
    {
        "messaggio": "Buongiorno, sono Marco Rossi e il mio numero è 333 1234567.",
        "note_operatore": "Cliente VIP, seguito da Stefano",
    },
    {
        "messaggio": "Potete mandare la fattura a giulia.bianchi@esempio.it? Grazie, Giulia Bianchi.",
        "note_operatore": "Verificare insoluto precedente",
    },
    {
        "messaggio": "Il mio codice fiscale è RSSMRA85M01H501Z, intestato a Mario Rossi.",
        "note_operatore": "Documento già validato",
    },
    {
        "messaggio": "Sono Elena Conti (CNTLNE90A41F205X), tel. +39 347 987 6543, email elena.conti@mail.it.",
        "note_operatore": "Richiesta urgente",
    },
    {
        "messaggio": "Chiamatemi in ufficio allo 02 55443322, chiedete di Luca Ferretti.",
        "note_operatore": "Centralino interno 402",
    },
    {
        "messaggio": "Ho cambiato cellulare, il nuovo è 348-7654321. Saluti, Alessandro Gallo.",
        "note_operatore": "Aggiornare anagrafica CRM",
    },
    {
        "messaggio": "Inviate il preventivo a francesco.moretti+preventivi@azienda.it entro oggi.",
        "note_operatore": "Sconto 10% approvato",
    },
    {
        "messaggio": "Delego al ritiro mia moglie Chiara Romano, codice fiscale RMNCHR88D52L219K.",
        "note_operatore": "Delega firmata allegata",
    },
    {
        "messaggio": "Per qualsiasi problema contattate Andrea Fontana o Sara Barbieri al 320 1122334.",
        "note_operatore": "Doppio referente",
    },
    {
        "messaggio": "Ecco i dati per il contratto: CF VRDLGU75C12A944P e mail luigi.verdi@pec.it.",
        "note_operatore": "Inviare contratto standard",
    },
    {
        "messaggio": "Abito a Torino, il mio fisso è 011 9876543. Mi chiamo Roberto Martini.",
        "note_operatore": "Zona copertura fibra ok",
    },
    {
        "messaggio": "Il codice fiscale scritto in minuscolo è bnchra92e45d612y, confermate?",
        "note_operatore": "Controllo formato CF",
    },
    {
        "messaggio": "Dall'estero uso 0039 339 4455667 oppure scrivete a paolo.neri@studio.org.",
        "note_operatore": "Cliente residente a Lugano",
    },
    {
        "messaggio": "Sono Davide Colombo, ho aperto il ticket #48291 per un addebito di 120 euro.",
        "note_operatore": "Rimborso non dovuto",
    },
    {
        "messaggio": "La pratica di Matteo Rinaldi (RNLMTT80B10G273S) è ancora bloccata.",
        "note_operatore": "Escalation livello 2",
    },
    {
        "messaggio": "Aggiornamento recapiti per Silvia De Luca: nuovo num 366.8899001, vecchia mail s.deluca@old.it.",
        "note_operatore": "Vecchio numero disattivato",
    },
    {
        "messaggio": "Richiesta informazioni da parte di Giorgio Marchetti, telefono 06 1234 5678.",
        "note_operatore": "Richiamare dopo le 15:00",
    },
    {
        "messaggio": "Vorrei sapere gli orari di apertura dello sportello nella giornata di venerdì.",
        "note_operatore": "Info generica",
    },
    {
        "messaggio": "Il pagamento per l'ordine 20269981 è andato a buon fine, grazie mille.",
        "note_operatore": "Nessuna azione richiesta",
    },
    {
        "messaggio": "Contatto emergenza: Federica Serra, CF SRRFRC95L60H501U, cell 340 5566778, federica@serra.it.",
        "note_operatore": "NOTA RISERVATA: cliente ha minacciato reclamo",
    },
]

for idx, record in enumerate(test_records, start=1):
    payload = build_gemini_payload(record)
    assert "note_operatore" not in payload, "ERRORE: le note operatore non devono passare!"
    print(f"--- Esempio {idx:02d} ---")
    print(f"PRIMA: {record['messaggio']}")
    print(f"DOPO : {payload['messaggio_anonimizzato']}\n")

--- Esempio 01 ---
PRIMA: Buongiorno, sono Marco Rossi e il mio numero è 333 1234567.
DOPO : Buongiorno, sono [NOME] e il mio numero è [TELEFONO].

--- Esempio 02 ---
PRIMA: Potete mandare la fattura a giulia.bianchi@esempio.it? Grazie, Giulia Bianchi.
DOPO : Potete mandare la fattura a [EMAIL]? Grazie, [NOME].

--- Esempio 03 ---
PRIMA: Il mio codice fiscale è RSSMRA85M01H501Z, intestato a Mario Rossi.
DOPO : Il mio codice fiscale è [CF], intestato a [NOME].

--- Esempio 04 ---
PRIMA: Sono Elena Conti (CNTLNE90A41F205X), tel. +39 347 987 6543, email elena.conti@mail.it.
DOPO : Sono [NOME] ([CF]), tel. [TELEFONO], email [EMAIL].

--- Esempio 05 ---
PRIMA: Chiamatemi in ufficio allo 02 55443322, chiedete di Luca Ferretti.
DOPO : Chiamatemi in ufficio allo [TELEFONO], chiedete di [NOME].

--- Esempio 06 ---
PRIMA: Ho cambiato cellulare, il nuovo è 348-7654321. Saluti, Alessandro Gallo.
DOPO : Ho cambiato cellulare, il nuovo è [TELEFONO]. Saluti, [NOME].

--- Esempio 07 ---
PRIMA: Inviate